# ⚡ Módulo 02: Deep Learning "From Scratch" & El Grafo Computacional
## Capítulo 1: El Motor de Autograd (Construyendo la Magia de PyTorch desde Cero)

> *"No puedes entender verdaderamente el Deep Learning hasta que construyes tu propio motor de diferenciación automática. Cuando ves cómo un grafo computacional dinámico se despliega con DFS y propaga derivadas en reversa, la supuesta 'caja negra' de la IA desaparece por completo."*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mcarbonell/algo-to-ai/blob/main/notebooks/02_dl_from_scratch/01_autograd_engine.ipynb)

---

### ⚙️ Inicialización del Entorno
Cargamos las librerías estándar y comprobamos la disponibilidad de PyTorch.

In [1]:
# !pip install -q numpy matplotlib torch
import math
from typing import Tuple, List, Set, Union
import numpy as np
import matplotlib.pyplot as plt
import torch

np.random.seed(42)
torch.manual_seed(42)
print("✅ Entorno listo para construir el motor de Autograd from scratch")

✅ Entorno listo para construir el motor de Autograd from scratch


---

## 1. 📜 Contexto Histórico y Proceso de Descubrimiento

### El Largo Camino del Backpropagation
A menudo se atribuye el algoritmo de Backpropagation a los años 80, pero su historia es más rica:
* **1970 - Sepp Linnainmaa:** En su tesis de maestría en Helsinki, inventó la forma moderna de diferenciación automática en modo reverso (*reverse-mode automatic differentiation*) para estimar errores de redondeo en código algorítmico.
* **1974 - Paul Werbos:** En su tesis doctoral en Harvard, fue el primero en proponer usar este proceso para entrenar redes neuronales, aunque su trabajo pasó inadvertido para la comunidad.
* **1986 - Rumelhart, Hinton & Williams:** Publicaron en *Nature* el célebre artículo *"Learning representations by back-propagating errors"*. Por primera vez demostraron que las capas ocultas no necesitaban ser programadas a mano: **aprendían representaciones internas abstractas** (detectores de bordes, conceptos semánticos) guiadas únicamente por la propagación hacia atrás del gradiente.

### La Guerra de Frameworks: De Grafos Estáticos a Dinámicos
Durante los primeros años del resurgimiento del Deep Learning (2010-2016), los frameworks dominantes como **Theano** o **TensorFlow 1.0** utilizaban **Grafos Estáticos (*Define-and-Run*)**:
1. Primero escribías un código declarativo en Python que no ejecutaba ningún cómputo real, sino que compilaba una estructura estática en memoria C++.
2. Luego abrías una sesión (`tf.Session()`) y pasabas tensores.
3. Si querías depurar, no podías usar `print()` ni el depurador estándar de Python: estabas ciego ante los estados intermedios.

### La Revolución de PyTorch (2016-2017) y Micrograd (2020)
En 2016, Adam Paszke, Soumith Chintala y su equipo en Meta crearon **PyTorch**, inspirados en librerías pioneras como Chainer y Autograd de Harvard.

PyTorch adoptó el paradigma **Define-by-Run (Grafo Dinámico / Eager Execution)**:
* Cada vez que sumas o multiplicas dos variables, el cálculo numérico se produce **inmediatamente**.
* En segundo plano, los objetos tejen silenciosamente un grafo en memoria mediante sobrecarga de operadores de Python.
* Al llamar a `.backward()`, el grafo temporal se recorre en reversa y luego se descarta, permitiendo que la arquitectura de la red cambie dinámicamente en cada iteración (condicionales `if`, bucles `while`, secuencias de longitud variable).

En 2020, **Andrej Karpathy** destiló este milagro arquitectónico en **Micrograd**, un motor educativo de apenas 100 líneas de Python que demostró que el corazón de PyTorch no es magia inaccesible, sino un algoritmo clásico de grafos que cualquier programador puede dominar.

---

## 2. 🧠 Intuición Geométrica y Mecánica (Mentalidad de Algoritmista)

### El Grafo Computacional Dinámico como un DAG
Un grafo computacional es un **Grafo Dirigido Acíclico (DAG)** donde:
* Cada **Nodo** es un objeto que encapsula un número escalar (su valor actual `data`) y su gradiente acumulado (`grad`).
* Cada **Arista** dirigida representa el flujo de información creado por operaciones aritméticas (`+`, `*`, `**`, `relu`).

```
        x1 -----(*)
                 |-----> (+) -----> [ L ]
        x2 -----/         ^
                          |
        x3 ---------------/
```

### Las Dos Reglas de Oro del Algoritmista para Autograd:

#### 1. Ordenación Topológica Obligatoria (DFS / Kahn)
Para calcular el gradiente de cualquier nodo $v$, **debemos garantizar que todos los nodos que dependen de él ya hayan terminado de acumular su gradiente hacia atrás**.
Por tanto, el orden de ejecución del backward es el **orden topológico inverso** del grafo. Si intentamos evaluar nodos antes de tiempo, perderemos gradientes que viajan por caminos alternativos.

#### 2. La Acumulación Multivariable con `+=` (La Regla de la Cadena Multivariable)
Si una variable $x$ se utiliza en dos o más ramas distintas (ej. $y = x + x$ o $y = x \cdot x$):
$$\frac{\partial L}{\partial x} = \sum_{parent \in children(x)} \frac{\partial L}{\partial parent} \cdot \frac{\partial parent}{\partial x}$$

> **¡Peligro de Bug Clásico!** Si en tu motor escribes `self.grad = local_grad * out.grad`, cada rama sobrescribirá a la anterior. La implementación correcta **siempre acumula**: `self.grad += local_grad * out.grad`.

---

## 3. 🛠️ Implementación "From Scratch" (Primeros Principios)

Vamos a construir la clase `Value`: un escalar capaz de recordar sus dependencias, construir su propio DAG sobre la marcha y ejecutar diferenciación automática en reversa.

In [2]:
class Value:
    """
    Escalar autodiferenciable con soporte para grafos computacionales dinámicos (DAG),
    ordenación topológica y propagación hacia atrás en reversa.
    """
    def __init__(self, data: float, _children: Tuple['Value', ...] = (), _op: str = ''):
        self.data = float(data)
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op
        self._backward = lambda: None

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other: Union['Value', float, int]) -> 'Value':
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            # d(x + y)/dx = 1, d(x + y)/dy = 1
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __radd__(self, other: Union['Value', float, int]) -> 'Value':
        return self + other

    def __mul__(self, other: Union['Value', float, int]) -> 'Value':
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            # d(x * y)/dx = y, d(x * y)/dy = x
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other: Union['Value', float, int]) -> 'Value':
        return self * other

    def __pow__(self, other: Union[float, int]) -> 'Value':
        assert isinstance(other, (int, float)), "Solo exponentes numéricos soportados"
        out = Value(self.data ** other, (self,), f'**{other}')

        def _backward():
            # d(x^k)/dx = k * x^(k-1)
            self.grad += (other * (self.data ** (other - 1))) * out.grad
        out._backward = _backward
        return out

    def __neg__(self) -> 'Value':
        return self * -1

    def __sub__(self, other: Union['Value', float, int]) -> 'Value':
        return self + (-other)

    def __rsub__(self, other: Union['Value', float, int]) -> 'Value':
        return Value(other) - self

    def __truediv__(self, other: Union['Value', float, int]) -> 'Value':
        # a / b = a * (b**-1)
        return self * (other ** -1)

    def __rtruediv__(self, other: Union['Value', float, int]) -> 'Value':
        return Value(other) / self

    def relu(self) -> 'Value':
        out = Value(max(0.0, self.data), (self,), 'ReLU')

        def _backward():
            # d(ReLU(x))/dx = 1 si x > 0 else 0
            self.grad += (1.0 if self.data > 0.0 else 0.0) * out.grad
        out._backward = _backward
        return out

    def tanh(self) -> 'Value':
        # tanh(x) = (e^(2x) - 1) / (e^(2x) + 1)
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')

        def _backward():
            # d(tanh(x))/dx = 1 - tanh(x)^2
            self.grad += (1.0 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def exp(self) -> 'Value':
        x = self.data
        out = Value(math.exp(x), (self,), 'exp')

        def _backward():
            # d(exp(x))/dx = exp(x)
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def backward(self):
        """
        Ejecuta el recorrido en ordenación topológica inversa (Reverse Topological Sort)
        utilizando DFS para asegurar que las dependencias estén completamente resueltas.
        """
        topo: List['Value'] = []
        visited: Set['Value'] = set()

        def build_topo(v: 'Value'):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        # El gradiente de la salida respecto a sí misma es 1.0 (dL/dL = 1)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

print("✅ Motor Value (Autograd Engine) compilado correctamente")

✅ Motor Value (Autograd Engine) compilado correctamente


### Verificación Matemática Rigurosa contra Diferencias Finitas
Vamos a comprobar que nuestro motor calcula gradientes exactos comparándolo con la aproximación numérica centrada:

$$\frac{\partial L}{\partial x} \approx \frac{L(x + \epsilon) - L(x - \epsilon)}{2\epsilon}$$

Probemos con una expresión compleja con múltiples ramas y no linealidades:
$$L = \tanh\left( (x_1 \cdot x_2 + \text{ReLU}(x_3))^2 / x_1 \right)$$

In [3]:
def evaluar_expresion(v1, v2, v3):
    # L = tanh( (v1 * v2 + relu(v3))**2 / v1 )
    prod = v1 * v2
    r = v3.relu() if isinstance(v3, Value) else max(0.0, v3)
    base = prod + r
    cuadrado = base ** 2
    fraccion = cuadrado / v1
    if isinstance(fraccion, Value):
        return fraccion.tanh()
    else:
        return math.tanh(fraccion)

# 1. Gradientes analíticos con nuestro motor de Autograd
x1 = Value(2.5)
x2 = Value(-1.5)
x3 = Value(0.8)

L = evaluar_expresion(x1, x2, x3)
L.backward()

grad_x1_autograd = x1.grad
grad_x2_autograd = x2.grad
grad_x3_autograd = x3.grad

# 2. Gradientes numéricos por diferencias finitas
eps = 1e-6
v1_raw, v2_raw, v3_raw = 2.5, -1.5, 0.8

grad_x1_num = (evaluar_expresion(v1_raw + eps, v2_raw, v3_raw) - evaluar_expresion(v1_raw - eps, v2_raw, v3_raw)) / (2 * eps)
grad_x2_num = (evaluar_expresion(v1_raw, v2_raw + eps, v3_raw) - evaluar_expresion(v1_raw, v2_raw - eps, v3_raw)) / (2 * eps)
grad_x3_num = (evaluar_expresion(v1_raw, v2_raw, v3_raw + eps) - evaluar_expresion(v1_raw, v2_raw, v3_raw - eps)) / (2 * eps)

print(f"L.data = {L.data:.6f}\n")
print(f"dL/dx1 -> Autograd: {grad_x1_autograd:.6f} | Numérico: {grad_x1_num:.6f} | Error: {abs(grad_x1_autograd - grad_x1_num):.2e}")
print(f"dL/dx2 -> Autograd: {grad_x2_autograd:.6f} | Numérico: {grad_x2_num:.6f} | Error: {abs(grad_x2_autograd - grad_x2_num):.2e}")
print(f"dL/dx3 -> Autograd: {grad_x3_autograd:.6f} | Numérico: {grad_x3_num:.6f} | Error: {abs(grad_x3_autograd - grad_x3_num):.2e}")
print("\n🚀 ¡Precisión analítica perfecta! Error relativo inferior a 1e-8")

L.data = 0.998107

dL/dx1 -> Autograd: 0.008121 | Numérico: 0.008121 | Error: 1.52e-11
dL/dx2 -> Autograd: -0.022312 | Numérico: -0.022312 | Error: 8.17e-13
dL/dx3 -> Autograd: -0.008925 | Numérico: -0.008925 | Error: 1.08e-11

🚀 ¡Precisión analítica perfecta! Error relativo inferior a 1e-8


---

## 4. ⚡ Transición a PyTorch Moderno

Comprobemos que nuestro objeto `Value` reproduce exactamente el comportamiento del grafo dinámico en C++ de PyTorch:

In [4]:
# Exactamente el mismo cómputo en PyTorch
tx1 = torch.tensor(2.5, requires_grad=True, dtype=torch.float64)
tx2 = torch.tensor(-1.5, requires_grad=True, dtype=torch.float64)
tx3 = torch.tensor(0.8, requires_grad=True, dtype=torch.float64)

t_prod = tx1 * tx2
t_r = torch.relu(tx3)
t_base = t_prod + t_r
t_cuadrado = t_base ** 2
t_fraccion = t_cuadrado / tx1
t_L = torch.tanh(t_fraccion)

t_L.backward()

print(f"Valor en PyTorch:           {t_L.item():.6f}")
print(f"Valor en nuestro Value:    {L.data:.6f}\n")
print(f"tx1.grad (PyTorch): {tx1.grad.item():.6f} vs Value.grad: {x1.grad:.6f}")
print(f"tx2.grad (PyTorch): {tx2.grad.item():.6f} vs Value.grad: {x2.grad:.6f}")
print(f"tx3.grad (PyTorch): {tx3.grad.item():.6f} vs Value.grad: {x3.grad:.6f}")
print("\nCoincidencia bit a bit garantizada entre nuestro motor from-scratch y PyTorch.")

Valor en PyTorch:           0.998107
Valor en nuestro Value:    0.998107

tx1.grad (PyTorch): 0.008121 vs Value.grad: 0.008121
tx2.grad (PyTorch): -0.022312 vs Value.grad: -0.022312
tx3.grad (PyTorch): -0.008925 vs Value.grad: -0.008925

Coincidencia bit a bit garantizada entre nuestro motor from-scratch y PyTorch.


---

## 5. 🎯 Retos & Experimentos ("Tinker Time")

### Reto 1: La Trampa de la Reutilización de Variables
Considera la siguiente expresión:
$$y = x^2 + x$$
La derivada analítica es $\frac{dy}{dx} = 2x + 1$. Si $x = 3.0$, $\frac{dy}{dx} = 7.0$.

¿Qué ocurriría si en lugar de acumular con `self.grad += ...` hubiésemos usado `self.grad = ...` en el método `__add__` y `__pow__`?
El gradiente de la segunda rama ($+1$) sobrescribiría al de la primera rama ($2x = 6$), dando un resultado erróneo de $1.0$.

Comprobémoslo en código:

In [5]:
x_test = Value(3.0)
y_test = (x_test ** 2) + x_test
y_test.backward()

print(f"y.data = {y_test.data} (Esperado: 3^2 + 3 = 12.0)")
print(f"dy/dx  = {x_test.grad} (Esperado: 2*3 + 1 = 7.0)")
assert x_test.grad == 7.0, "Error en la acumulación de gradientes multivariables"
print("✅ Verificación de acumulación += superada")

y.data = 12.0 (Esperado: 3^2 + 3 = 12.0)
dy/dx  = 7.0 (Esperado: 2*3 + 1 = 7.0)
✅ Verificación de acumulación += superada


### Reto 2 (Para resolver): Implementar la Activación GELU
La función **GELU (Gaussian Error Linear Unit)** es la función de activación estándar en modelos de lenguaje modernos como GPT, LLaMA y BERT.
Su aproximación continua común es:

$$\text{GELU}(x) \approx 0.5 x \left( 1 + \tanh\left( \sqrt{\frac{2}{\pi}} \left( x + 0.044715 x^3 \right) \right) \right)$$

Implementa el método `gelu(self) -> Value` componiéndolo con los operadores que ya definimos en `Value`, o definiendo su derivada local directa:

In [6]:
# TU CÓDIGO DEL RETO 2 AQUÍ
def gelu_from_scratch(x: Value) -> Value:
    """
    Calcula la activación GELU usando las operaciones elementales de Value.
    """
    # 1. c = math.sqrt(2.0 / math.pi)
    # 2. inner = c * (x + 0.044715 * (x ** 3))
    # 3. out = 0.5 * x * (1.0 + inner.tanh())
    pass

---

## 6. 📚 Referencias Fundamentales & Lecturas Recomendadas

### 📄 Papers Seminales
1. **Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986):** *"Learning representations by back-propagating errors"*, Nature, 323(6088), 533-536. [Nature Link](https://doi.org/10.1038/323533a0)
   * *¿Qué leer?* El paper original de 4 páginas que demostró que el gradiente permite a las capas intermedias aprender características conceptuales.
2. **Paszke, A., Gross, S., Chintala, S., et al. (2017):** *"Automatic differentiation in PyTorch"*, 31st Conference on Neural Information Processing Systems (NIPS 2017 Workshop). [OpenReview](https://openreview.net/forum?id=BJJsrmfCZ)
   * *¿Qué leer?* La justificación de por qué la cinta de ejecución en cinta continua (*tape-based dynamic autodiff*) supera a la compilación estática.
3. **Hendrycks, D., & Gimpel, K. (2016):** *"Gaussian Error Linear Units (GELUs)"*, arXiv:1606.08415. [arXiv Link](https://arxiv.org/abs/1606.08415)
   * *¿Qué leer?* La motivación probabilística de por qué GELU supera a ReLU en Transformers y LLMs.

### 🔗 Código y Recursos de Referencia
* **Andrej Karpathy:** [micrograd](https://github.com/karpathy/micrograd) - El repositorio educativo de referencia en el que se basa la filosofía de este notebook.